#### Pipeline


Data Cleaning -> Rule Based Approach -> Nlp Approach -> Brand Name  

In [64]:
import pandas as pd

test_df = pd.read_csv("/Users/omkar/Documents/SmartShop/app/sains_food_cupboard.csv")

In [151]:
test_df.columns

Index(['supermarket', 'prices_(£)', 'prices_unit_(£)', 'unit', 'names', 'date',
       'category', 'own_brand'],
      dtype='str')

In [55]:
test_df.head()
prd_names = list(test_df[(test_df['own_brand'] == False)]['names'].unique())

In [56]:
len(prd_names)

6262

In [57]:
prd_names[:10]

['Maryland Cookies Chocolate Chip Minis x6',
 'Weetabix Cereal x24',
 "Walker's Shortbread Fingers x10 160g",
 'Stamford Street Co. Chopped Tomatoes in Tomato Juice 400g',
 'Cravendale Filtered Fresh Semi Skimmed Milk 2L Fresher for Longer',
 'JS Double Cream 300ml',
 'Heinz Baked Beans in a Rich Tomato Sauce 4 x 415g',
 'Walkers Ready Salted Multipack Crisps 6x25g',
 'Napolina Chopped Tomatoes 4x400g',
 'Napolina Chopped Tomatoes 400g']

In [2]:
import re

def clean_text(text):
    text = re.sub(r'\d+g|\d+kg|\d+ml|\d+L', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

In [49]:
KNOWN_BRANDS = ["ASDA","Sainsburys","Tesco","Morrisons"]

In [14]:
def extract_brand_hybrid(text):
    # Rule-based first
    for brand in KNOWN_BRANDS:
        if text.startswith(brand):
            return brand
    
    # Fallback to NLP
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ == "ORG":
            return ent.text
    
    return "Unknown"

In [22]:
for name in prd_names[:10]:  # Test on first 100 names
    cleaned_name = clean_text(name)
    brand = extract_brand_hybrid(cleaned_name)
    print(f"| Cleaned: {cleaned_name} | Brand: {brand}")

| Cleaned: Maryland Cookies Chocolate Chip Minis x6 | Brand: Maryland Cookies Chocolate Chip Minis
| Cleaned: Weetabix Cereal x24 | Brand: Unknown
| Cleaned: Walkers Shortbread Fingers x10 | Brand: Unknown
| Cleaned: Sainsburys British Semi Skimmed Milk 2 4 pint | Brand: Sainsburys
| Cleaned: Sainsburys Fairtrade Bananas x5 | Brand: Sainsburys
| Cleaned: Sainsburys Red Seedless Grapes | Brand: Sainsburys
| Cleaned: Sainsburys British Free Range Eggs Large x12 | Brand: Sainsburys
| Cleaned: Sainsburys Easy Peelers Taste the Difference | Brand: Sainsburys
| Cleaned: Stamford Street Co Chopped Tomatoes in Tomato Juice | Brand: Stamford Street Co
| Cleaned: Sainsburys British Butter Salted | Brand: Sainsburys


### Extracting the package size from the name.


In [143]:
df = test_df[['names','own_brand']].copy()
df.head()

,names,own_brand
0,Maryland Cookies Chocolate Chip Minis x6,False
1,Weetabix Cereal x24,False
2,Walker's Shortbread Fingers x10 160g,False
3,Sainsbury's British Semi Skimmed Milk 2.27L (4...,True
4,Sainsbury's Fairtrade Bananas x5,True


In [144]:
import re

def clean_product_name(text):
    text = text.lower()

    # Remove patterns like "2 x 500ml"
    text = re.sub(r'\d+\s*[xX]\s*\d*\.?\d+\s*(kg|g|l|ml)', '', text)

    # Remove single weights like "500g", "1.5l"
    text = re.sub(r'\d*\.?\d+\s*(kg|g|l|ml)', '', text)

    # Remove pack info like "x6", "pack of 6"
    text = re.sub(r'(x|pack of)\s*\d+', '', text)

    # Clean extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    text = text.replace('(', '').replace(')', '')

    return text

In [145]:
df['clean_name'] = df['names'].apply(lambda x: clean_product_name(x))

In [146]:
df.head()

,names,own_brand,clean_name
0,Maryland Cookies Chocolate Chip Minis x6,False,maryland cookies chocolate chip minis
1,Weetabix Cereal x24,False,weetabix cereal
2,Walker's Shortbread Fingers x10 160g,False,walker's shortbread fingers
3,Sainsbury's British Semi Skimmed Milk 2.27L (4...,True,sainsbury's british semi skimmed milk 4 pint
4,Sainsbury's Fairtrade Bananas x5,True,sainsbury's fairtrade bananas


In [147]:
df['pack_info'] = df['names'].str.findall(r'\b\w*\d\w*\b')

In [148]:
df.head()

,names,own_brand,clean_name,pack_info
0,Maryland Cookies Chocolate Chip Minis x6,False,maryland cookies chocolate chip minis,[x6]
1,Weetabix Cereal x24,False,weetabix cereal,[x24]
2,Walker's Shortbread Fingers x10 160g,False,walker's shortbread fingers,"[x10, 160g]"
3,Sainsbury's British Semi Skimmed Milk 2.27L (4...,True,sainsbury's british semi skimmed milk 4 pint,"[2, 27L, 4]"
4,Sainsbury's Fairtrade Bananas x5,True,sainsbury's fairtrade bananas,[x5]


In [149]:
df['extracted_pack_info'] = df['names'].str.split().apply(lambda words: [w for w in words if re.search(r'\d', w)])

In [150]:
df

,names,own_brand,clean_name,pack_info,extracted_pack_info
0,Maryland Cookies Chocolate Chip Minis x6,False,maryland cookies chocolate chip minis,[x6],[x6]
1,Weetabix Cereal x24,False,weetabix cereal,[x24],[x24]
2,Walker's Shortbread Fingers x10 160g,False,walker's shortbread fingers,"[x10, 160g]","[x10, 160g]"
3,Sainsbury's British Semi Skimmed Milk 2.27L (4...,True,sainsbury's british semi skimmed milk 4 pint,"[2, 27L, 4]","[2.27L, (4]"
4,Sainsbury's Fairtrade Bananas x5,True,sainsbury's fairtrade bananas,[x5],[x5]
...,...,...,...,...,...
508709,Reese's Peanut Butter Crème Egg 5 x 34g (170g),False,reese's peanut butter crème egg,"[5, 34g, 170g]","[5, 34g, (170g)]"
508710,Sainsbury's Fig Rolls 200g,True,sainsbury's fig rolls,[200g],[200g]
508711,Sainsbury's Conchiglie (Shells) 500g,True,sainsbury's conchiglie shells,[500g],[500g]
508712,Paxo Veggie Fillers Tomato & Herb 120g,False,paxo veggie fillers tomato & herb,[120g],[120g]


### Extracting using the naming convention
[Brand] + [Descriptor(s)] + [Product Type] + [Variant] + [Size/Unit]

In [161]:
### Introducing brand column and replacing know brands wih their names and unknown with "Unknown".

df['own_brand'].where(df['own_brand'] == True, "sainsburys")

0         sainsburys
1         sainsburys
2         sainsburys
3               True
4               True
             ...    
508709    sainsburys
508710          True
508711          True
508712    sainsburys
508713          True
Name: own_brand, Length: 508714, dtype: object

In [163]:
df[df['own_brand'] == True]

,names,own_brand,clean_name,pack_info,extracted_pack_info
3,Sainsbury's British Semi Skimmed Milk 2.27L (4...,True,sainsbury's british semi skimmed milk 4 pint,"[2, 27L, 4]","[2.27L, (4]"
4,Sainsbury's Fairtrade Bananas x5,True,sainsbury's fairtrade bananas,[x5],[x5]
5,Sainsbury's Red Seedless Grapes 500g,True,sainsbury's red seedless grapes,[500g],[500g]
6,Sainsbury's British Free Range Eggs Large x12,True,sainsbury's british free range eggs large,[x12],[x12]
7,"Sainsbury's Easy Peelers, Taste the Difference...",True,"sainsbury's easy peelers, taste the difference",[600g],[600g]
...,...,...,...,...,...
508707,Sainsbury's Roasted Salted Pistachios 300g,True,sainsbury's roasted salted pistachios,[300g],[300g]
508708,Sainsbury's Roasted Pistachios in Shells 270g,True,sainsbury's roasted pistachios in shells,[270g],[270g]
508710,Sainsbury's Fig Rolls 200g,True,sainsbury's fig rolls,[200g],[200g]
508711,Sainsbury's Conchiglie (Shells) 500g,True,sainsbury's conchiglie shells,[500g],[500g]


In [167]:
df['own_brand'].value_counts()

own_brand
False    373525
True     135189
Name: count, dtype: int64

In [173]:
df['brand'] = df['own_brand'].replace({True: "sainsburys", False: "Unknown"})

In [174]:
df.head()

,names,own_brand,clean_name,pack_info,extracted_pack_info,brand
0,Maryland Cookies Chocolate Chip Minis x6,False,maryland cookies chocolate chip minis,[x6],[x6],Unknown
1,Weetabix Cereal x24,False,weetabix cereal,[x24],[x24],Unknown
2,Walker's Shortbread Fingers x10 160g,False,walker's shortbread fingers,"[x10, 160g]","[x10, 160g]",Unknown
3,Sainsbury's British Semi Skimmed Milk 2.27L (4...,True,sainsbury's british semi skimmed milk 4 pint,"[2, 27L, 4]","[2.27L, (4]",sainsburys
4,Sainsbury's Fairtrade Bananas x5,True,sainsbury's fairtrade bananas,[x5],[x5],sainsburys


In [175]:
df['brand'].value_counts()

brand
Unknown       373525
sainsburys    135189
Name: count, dtype: int64

In [41]:
for name in prd_names[:10]:  # Test on first 100 names
    cleaned_name = clean_text(name).split()[:3]
    print(cleaned_name)
    

['Maryland', 'Cookies', 'Chocolate']
['Weetabix', 'Cereal', 'x24']
['Walkers', 'Shortbread', 'Fingers']
['Sainsburys', 'British', 'Semi']
['Sainsburys', 'Fairtrade', 'Bananas']
['Sainsburys', 'Red', 'Seedless']
['Sainsburys', 'British', 'Free']
['Sainsburys', 'Easy', 'Peelers']
['Stamford', 'Street', 'Co']
['Sainsburys', 'British', 'Butter']


In [35]:
len(prd_names[1].split())

3

In [189]:
df[df['own_brand'] == False]['clean_name'].str.split().apply(lambda x: x[:1]).astype(str)

0           ['maryland']
1           ['weetabix']
2           ["walker's"]
8           ['stamford']
12        ['cravendale']
               ...      
508703        ['truvia']
508704      ['ambrosia']
508706        ['quaker']
508709       ["reese's"]
508712          ['paxo']
Name: clean_name, Length: 373525, dtype: str

In [196]:
df[df['own_brand'] == False]['clean_name'] \
    .str.split() \
    .str[0:3].value_counts().head(20)

clean_name
[cadbury, dairy, milk]         3250
[stamford, street, co.]        3138
[old, el, paso]                3037
[quaker, oat, so]              2524
[tilda, microwave, rice]       1712
[the, spice, tailor]           1603
[batchelors, cup, a]           1551
[heinz, baked, beans]          1445
[taylors, of, harrogate]       1126
[pip, &, nut]                  1053
[lee, kum, kee]                1036
[hartley's, 10, cal]            959
[batchelors, pasta, 'n']        917
[kellogg's, special, k]         881
[kellogg's, rice, krispies]     873
[john, west, tuna]              851
[lindt, excellence, dark]       842
[heinz, cream, of]              822
[knorr, stock, pot]             800
[princes, tuna, chunks]         792
Name: count, dtype: int64

#### Hybrid System 

- Dictionary match (fast win):
    
        If match => return

- ML/NER model: 

        If no match  => predict
- Fallback rule:

        First 1–2 tokens

### Creating a Dictionary:

To Create a Dictionary We need to Extract the common known brands manually to create additional data.

In [194]:
df[df['own_brand'] == False]['clean_name'] \
    .str.split() \
    .str[0].nunique()

942

There are 942 unique brands in Food Cupboard Data Itself

In [197]:
#### Generating n-gramns:
from collections import Counter

def generate_ngrams(names, n):
    ngrams = []
    
    for name in names:
        tokens = name.split()
        if len(tokens) >= n:
            ngrams.append(" ".join(tokens[:n]))
    
    return ngrams

In [198]:
#Count frequency
names = df['clean_name'].dropna().str.lower()

one_word = Counter(generate_ngrams(names, 1))
two_word = Counter(generate_ngrams(names, 2))
three_word = Counter(generate_ngrams(names, 3))

In [209]:
# Inspect top candidates
one_word.most_common(35)

[("sainsbury's", 135189),
 ('heinz', 11721),
 ('cadbury', 10490),
 ('lindt', 5558),
 ('walkers', 5339),
 ("kellogg's", 5226),
 ('batchelors', 5059),
 ('twinings', 4567),
 ("mcvitie's", 4151),
 ('john', 4073),
 ('dr.', 4034),
 ('the', 3981),
 ("hartley's", 3799),
 ('quaker', 3549),
 ('baxters', 3276),
 ('old', 3243),
 ('stamford', 3188),
 ('princes', 3150),
 ("jacob's", 3107),
 ('fudco', 2890),
 ('ambrosia', 2866),
 ('tilda', 2766),
 ("nando's", 2591),
 ('dolmio', 2527),
 ('napolina', 2411),
 ('extra', 2389),
 ('loyd', 2297),
 ("colman's", 2138),
 ('knorr', 2137),
 ("sharwood's", 2082),
 ('bonne', 2007),
 ('nescafé', 1999),
 ('kinder', 1978),
 ('starbucks', 1950),
 ('graze', 1946)]

In [203]:
two_word.most_common(20)

[('john west', 4073),
 ('dr. oetker', 3942),
 ('cadbury dairy', 3250),
 ('stamford street', 3188),
 ('old el', 3037),
 ('quaker oat', 2889),
 ("sainsbury's fairtrade", 2778),
 ('loyd grossman', 2297),
 ('bonne maman', 2007),
 ('tilda microwave', 1858),
 ("tony's chocolonely", 1800),
 ("sainsbury's deliciously", 1784),
 ("sainsbury's ground", 1763),
 ("sainsbury's sweet", 1730),
 ("sainsbury's tomato", 1682),
 ("sainsbury's ready", 1615),
 ('the spice', 1603),
 ('lindt lindor', 1585),
 ('heinz baked', 1579),
 ("sainsbury's italian", 1574)]

In [204]:
three_word.most_common(20)

[('cadbury dairy milk', 3250),
 ('stamford street co.', 3138),
 ('old el paso', 3037),
 ('quaker oat so', 2524),
 ("sainsbury's deliciously free", 1784),
 ('tilda microwave rice', 1712),
 ('the spice tailor', 1603),
 ('batchelors cup a', 1551),
 ('heinz baked beans', 1445),
 ("sainsbury's free from", 1375),
 ("sainsbury's ready to", 1260),
 ('taylors of harrogate', 1126),
 ("sainsbury's milk chocolate", 1063),
 ('pip & nut', 1053),
 ('lee kum kee', 1036),
 ("hartley's 10 cal", 959),
 ("batchelors pasta 'n'", 917),
 ("sainsbury's tomato &", 908),
 ("kellogg's special k", 881),
 ("kellogg's rice krispies", 873)]

In [210]:
## Keep Top N (filter noise)
TOP_N = 3

top_1 = set([x[0] for x in one_word.most_common(TOP_N)])
top_2 = set([x[0] for x in two_word.most_common(TOP_N)])
top_3 = set([x[0] for x in three_word.most_common(TOP_N)])

In [224]:
one_word.get("dr.")

4034

In [225]:
two_word.get("dr. karg")

In [228]:
two_word.get("dr. karg's")

92

In [227]:
df[df['clean_name'].str.startswith('dr')]['clean_name'].value_counts().head(20)

clean_name
dr. oetker milk chocolate chip chunks                92
dr. oetker 70% extra dark chocolate chip chunks      92
dr. oetker platinum grade leaf gelatine              92
dr. oetker gelatine sachets                          92
dr. oetker ready to roll white soft fondant icing    92
dr. oetker ready to roll fondant icing               92
dr. oetker extra strong red food colouring gel       92
dr. oetker extra strong green food colouring gel     92
dr. oetker extra strong blue food colouring gel      92
dr. karg's pumpkin seeds protein thins               92
dr. oetker extra strong orange food colouring gel    92
dr. oetker giant chocolate stars                     92
dr. oetker rainbow decorating icing                  92
dr. oetker extra strong pink food colouring gel      92
dr. oetker vege-gel gelatine sachets                 92
dr. oetker extra strong yellow food colouring gel    92
dr. oetker extra strong violet food colouring gel    92
dr. oetker extra strong black food co

## Selection Logic

In [229]:
from collections import defaultdict

prefix_next_map = {
    1: defaultdict(list),
    2: defaultdict(list),
    3: defaultdict(list)
}

names = df['clean_name'].dropna().str.lower()

for name in names:
    tokens = name.split()
    
    for n in [1, 2, 3]:
        if len(tokens) > n:
            prefix = " ".join(tokens[:n])
            next_token = tokens[n]
            
            prefix_next_map[n][prefix].append(next_token)



In [230]:
import pandas as pd

def compute_scores(prefix_map):
    data = []
    
    for n in prefix_map:
        for prefix, next_tokens in prefix_map[n].items():
            freq = len(next_tokens)
            diversity = len(set(next_tokens))
            
            score = freq * (diversity + 1)  # simple scoring
            
            data.append({
                "prefix": prefix,
                "n": n,
                "freq": freq,
                "diversity": diversity,
                "score": score
            })
    
    return pd.DataFrame(data)

score_df = compute_scores(prefix_next_map)

In [264]:
score_df.sort_values(by="percentile", ascending=False).head(25)

,prefix,n,freq,diversity,score,percentile
3,sainsbury's,1,135189,668,90441441,100.000000
7,heinz,1,11721,54,644655,99.988676
22,cadbury,1,10490,52,555970,99.977353
167,twinings,1,4567,38,178113,99.966029
8,walkers,1,5339,27,149492,99.954705
1714,dr. oetker,2,3942,35,141912,99.943381
291,fudco,1,2890,48,141610,99.932058
45,kellogg's,1,5226,23,125424,99.920734
13,mcvitie's,1,4151,29,124530,99.909410
4305,stamford street co.,3,3138,34,109830,99.898086


In [258]:
score_df['percentile'] = score_df['score'].rank(pct=True)
score_df['percentile'] = score_df['percentile'] * 100

In [272]:
score_df['percentile_within_n'] = score_df.groupby('n')['score'].rank(pct=True)

In [275]:
score_df[(score_df['percentile_within_n'] > 0.9) & (score_df['diversity'] > 15)]

,prefix,n,freq,diversity,score,percentile,percentile_within_n
3,sainsbury's,1,135189,668,90441441,100.000000,1.000000
7,heinz,1,11721,54,644655,99.988676,0.998937
8,walkers,1,5339,27,149492,99.954705,0.995749
9,napolina,1,2411,19,48220,99.762201,0.984060
11,alpro,1,1250,21,27500,99.524403,0.970244
12,ktc,1,1270,24,31750,99.614993,0.973433
13,mcvitie's,1,4151,29,124530,99.909410,0.992561
14,princes,1,3150,16,53550,99.784849,0.985122
16,jacob's,1,3107,18,59033,99.796173,0.986185
22,cadbury,1,10490,52,555970,99.977353,0.997875


In [278]:
def select_non_overlapping_brands(score_df):
    df = score_df.copy()
    
    # use strongest signal
    df = df.sort_values(by='percentile', ascending=False)
    
    selected = []
    
    for _, row in df.iterrows():
        candidate = row['prefix']
        
        # check if already covered by higher ranked brand
        if any(
            candidate == s or candidate.startswith(s) or s.startswith(candidate)
            for s in selected
        ):
            continue
        
        selected.append(candidate)
    
    return selected

In [279]:
result = select_non_overlapping_brands(score_df)

for r in result:
    print(r)

sainsbury's
heinz
cadbury
twinings
walkers
dr. oetker
fudco
kellogg's
mcvitie's
stamford street co.
lindt
john west
baxters
batchelors
the
hartley's
jacob's
princes
old el paso
napolina
natco
cofresh
ambrosia
starbucks
sharwood's
odysea
haribo
bonne maman
whitworths
blue dragon
colman's
ktc
graze
nando's
tilda microwave rice
merchant gourmet
loyd grossman
alpro
extra
rowse
nairn's
homepride
mutti
patak's
oxo
knorr
yutaka
ben's original
betty crocker
kinder
quaker
lavazza
lee kum kee
hellmann's
branston
itsu
kenco
dunn's river
dolmio
doritos
bisto
pot noodle
fox's
maltesers
weetabix
nescafé
capsicana
nomo
grace
tyrrells
jordans
kallo
tony's chocolonely
galaxy
wright's
kp
taylors of harrogate
naked
maggi
reese's
nissin soba
laila
dole
mccoy's
pringles
kitkat
sauce shop
aero
eat natural
belvita
nature valley
nakd
sacla
billington's
sarson's
bear
del monte
dorset cereals
forest feast
filippo berio
ryvita
organix
smarties
nature's finest
marmite
maynards bassetts
nestle
de cecco
bahlsen
mil

In [283]:
def grouped_clean_output(score_df):
    selected = select_non_overlapping_brands(score_df)
    
    output = {"1-word": [], "2-word": [], "3-word": []}
    
    for s in selected:
        n = len(s.split())
        output[f"{n}-word"].append(s)
    
    return output

gco = grouped_clean_output(score_df)

In [307]:
"dr. karg" in gco.get("2-word")

False

In [309]:
#### Creating a final list of brands to test on the data
final_brands = set(gco.get("1-word", []) + gco.get("2-word", []) + gco.get("3-word", []))   

In [321]:
def fill_brand(row, brands_set):
    if row['brand'] != 'Unknown':
        return row['brand']
    
    text = row['clean_name'].lower()
    
    # check multi-word first (important)
    for brand in sorted(brands_set, key=lambda x: -len(x.lower().split())):
        if brand in text:
            return brand
    
    return 'unknown'

In [323]:
brands_set = set(final_brands)
df['own_brand'] = df.apply(lambda row: fill_brand(row, brands_set), axis=1)

In [328]:
df.to_csv("/Users/omkar/Documents/SmartShop/notebook/nlp_output.csv", index=False)

In [327]:
df['own_brand'].value_counts()

own_brand
sainsburys                 135405
up                          18468
sweet                        9062
coffee                       8259
crisp                        7172
                            ...  
ben &                           5
j-basket seaweed crisps         5
rhythmluten free                5
baking buddy                    4
mr freeze sugar                 3
Name: count, Length: 931, dtype: int64

In [335]:
bad_words = {"in", "of", "to", "for", "and", "with","up",'cookies'}

brands_set = {b for b in brands_set if b not in bad_words and len(b) > 2}

In [346]:
### Cleaning the text.

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text

cleaned = clean_text(df['clean_name'].iloc[0])
print(cleaned)

maryland cookies chocolate chip minis


In [ ]:
score_map = dict(zip(score_df['prefix'], score_df['score']))
score_map

In [347]:
def get_all_matches(text, brands_set):
    matches = []
    
    for brand in brands_set:
        pattern = r'\b' + re.escape(brand) + r'\b'
        
        if re.search(pattern, text):
            matches.append(brand)
    
    return matches

matches = get_all_matches(cleaned, brands_set)
print(matches)

['maryland']


In [350]:
for m in matches:
    print(m, score_map.get(m, 0))

maryland 2155


In [351]:
def filter_matches(matches):
    return [m for m in matches if len(m) > 2]

filtered = filter_matches(matches)
print(filtered)

['maryland']


In [352]:
def get_position(text, brand):
    return text.find(brand)

for m in matches:
    print(m, get_position(cleaned, m))

maryland 0


In [353]:
def select_best_match(matches, text, score_map):
    if not matches:
        return None
    
    def score_fn(m):
        position = text.find(m)
        score = score_map.get(m, 0)
        
        return (
            -position,          # earlier is better
            score,              # higher is better
            len(m.split())      # longer is better
        )
    
    return max(matches, key=score_fn)

best = select_best_match(matches, cleaned, score_map)
print(best)

maryland


In [355]:
##### Debuging function

def debug_brand_extraction(text, brands_set, score_map):
    if not isinstance(text, str):
        return None, 0
    
    cleaned = clean_text(text)
    
    matches = get_all_matches(cleaned, brands_set)
    
    if not matches:
        return None, 0
    
    best = select_best_match(matches, cleaned, score_map)
    
    score = score_map.get(best, 0)
    
    return best, score

In [356]:
debug_df = df[['clean_name']].copy()

debug_df[['matched_brand', 'brand_score']] = debug_df['clean_name'].apply(
    lambda x: pd.Series(debug_brand_extraction(x, brands_set, score_map))
)

KeyboardInterrupt: 

In [ ]:
debug_df.head(20)

In [1]:
from nltk import pos_tag, word_tokenize
pos_tag(word_tokenize("John's big idea isn't all that bad.")) 

LookupError: 
**********************************************************************
  Resource 'punkt_tab' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'tokenizers/punkt_tab/english/'

  Searched in:
    - '/Users/omkar/nltk_data'
    - '/Users/omkar/Documents/SmartShop/.venv/nltk_data'
    - '/Users/omkar/Documents/SmartShop/.venv/share/nltk_data'
    - '/Users/omkar/Documents/SmartShop/.venv/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************
